# 第 52 章：Capstone 课程包目录页与发布版本说明

这个 notebook 用一个极小的 course-package directory table，对比“只按下一版优先级排序”的 naive baseline 和“先检查 release note、版本指针、读者说明与目录条目”的目录页 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_course_package_directory_release_notes_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['next_version_score'] = float(row['next_version_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} directory entries from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Target reader:', ordered_counts(rows, 'target_reader'))
print('Directory status:', ordered_counts(rows, 'directory_status'))
print('Release note status:', ordered_counts(rows, 'release_note_status'))
print('Version pointer status:', ordered_counts(rows, 'version_pointer_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['entry_id']
    for row in rows
    if row['reference_route'] == 'directory_ready'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['next_version_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'directory_ready'
    for row in top4_priority
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'directory_ready'
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_priority)

print('Reference directory-ready items:', reference_ready_ids)
print('Top-4 by next-version priority:')
for row in top4_priority:
    print(
        f"  {row['entry_id']}: priority={row['next_version_score']:.2f}, "
        f"route={row['reference_route']}, scope={row['entry_scope']}"
    )
print('Baseline directory precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_directory_item(row):
    if row['release_note_status'] in {'hold', 'blocked'}:
        return 'hold_for_next_version', 'release note state says this entry should wait for the next version'
    if row['version_pointer_status'] != 'synced':
        return 'sync_release_note_pointer', 'release-note pointer, version link, or summary is not synced'
    if row['audience_copy_status'] != 'clear':
        return 'rewrite_audience_copy', 'reader-facing summary is not clear enough yet'
    if row['directory_status'] != 'complete':
        return 'complete_directory_entry', 'directory entry, pointer, or one-line summary is incomplete'
    return 'directory_ready', 'entry has a complete directory row, synced version pointer, and clear audience copy'


routed_rows = []
for row in rows:
    route, reason = route_directory_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Course-package directory workflow routes:')
for row in routed_rows:
    print(
        f"  {row['entry_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'directory_ready']
entry_queue = [row for row in routed_rows if row['route'] == 'complete_directory_entry']
pointer_queue = [row for row in routed_rows if row['route'] == 'sync_release_note_pointer']
copy_queue = [row for row in routed_rows if row['route'] == 'rewrite_audience_copy']
hold_queue = [row for row in routed_rows if row['route'] == 'hold_for_next_version']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'directory_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Directory-ready queue:', [row['entry_id'] for row in ready_queue])
print('Complete-entry queue:', [row['entry_id'] for row in entry_queue])
print('Sync-pointer queue:', [row['entry_id'] for row in pointer_queue])
print('Rewrite-copy queue:', [row['entry_id'] for row in copy_queue])
print('Hold-next-version queue:', [row['entry_id'] for row in hold_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured directory precision:', round(ready_precision, 3))


In [ ]:
def directory_steps(row):
    steps = []
    if row['release_note_status'] in {'hold', 'blocked'}:
        steps.append('先决定这一项是否延期到下一版，并写清 hold 原因和解除条件。')
    if row['version_pointer_status'] != 'synced':
        steps.append('同步 release note、版本号、目录链接和更新摘要。')
    if row['audience_copy_status'] != 'clear':
        steps.append('重写读者说明：这是什么、谁该看它、下一步去哪。')
    if row['directory_status'] != 'complete':
        steps.append('补完整目录条目：标题、入口、版本、已知 caveat 和相关入口。')
    return steps or ['可以进入 course package directory；发布时保留版本号和最后更新时间。']


for row in routed_rows:
    if row['route'] != 'directory_ready':
        print(f"\n{row['entry_id']} -> {row['route']} ({row['entry_scope']})")
        for step in directory_steps(row):
            print(' -', step)


In [ ]:
directory_outline = [
    'Entry name: student bundle, TA bundle, instructor bundle, public archive, or maintenance pack',
    'Target reader: who should click this first',
    'One-line summary: what problem this entry solves',
    'Version pointer: current version, release note, and last update date',
    'Known caveat: what is deferred, held, or intentionally excluded',
    'Next step: where the reader should go after this entry',
]

directory_assistant_prompt = '''你是我的 capstone 课程包目录页助手。

任务：
1. 先阅读 course-package directory table，不要只按 next-version priority 排序；
2. 先检查 release note 状态；
3. 再检查 version pointer、audience copy 和 directory entry；
4. 把每个入口 route 到 directory_ready、complete_directory_entry、
   sync_release_note_pointer、rewrite_audience_copy 或 hold_for_next_version；
5. 对每个非 ready 入口输出目录发布前 checklist。

输出格式：
- Directory-ready entries
- Complete-entry entries
- Sync-pointer entries
- Rewrite-copy entries
- Hold-next-version entries
- Directory checklist by entry
'''

print('Course package directory outline:')
for item in directory_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(directory_assistant_prompt)


In [ ]:
directory_snapshot = {
    'dataset': DATA_PATH.name,
    'n_entries': len(rows),
    'baseline_top4_directory_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'directory_ready': [row['entry_id'] for row in ready_queue],
    'complete_directory_entry': [row['entry_id'] for row in entry_queue],
    'sync_release_note_pointer': [row['entry_id'] for row in pointer_queue],
    'rewrite_audience_copy': [row['entry_id'] for row in copy_queue],
    'hold_for_next_version': [row['entry_id'] for row in hold_queue],
    'python_version': platform.python_version(),
}

print('Course package directory snapshot:')
for key, value in directory_snapshot.items():
    print(f'  {key}: {value}')
